### **Prompt Chaining**

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
import os
import pprint

if os.getenv("GROQ_API_KEY") is None:
    raise ValueError("GROQ_API_KEY environment variable not set")


llm = ChatGroq(model= "meta-llama/llama-4-scout-17b-16e-instruct", temperature=0.5)
eval = ChatGroq(model= "openai/gpt-oss-120b", temperature=0.5)

- **State**

In [ ]:
class Blog(TypedDict):
    topic: str
    outline: str
    blog: str
    evaluation: str


- **Nodes**

In [ ]:
def create_outline(state: Blog)-> Blog:
    topic = state['topic']
    outline = llm.invoke(f"Create a detailed outline for a blog post about {topic}").content
    state['outline'] = outline
    return state

def write_blog(state: Blog)-> Blog:
    outline = state['outline']
    blog = llm.invoke(f"Write a blog post based on the following outline: {outline}").content
    state['blog'] = blog
    return state

def evaluate_blog(state: Blog)-> Blog:
    outline = state['outline']
    blog = state['blog']
    evaluation = eval.invoke(f"Evaluate the following blog post: {blog} for quality and accuracy and only give the number out of 10 by analysing the outline from which the blog made: {outline}").content
    state['evaluation'] = evaluation
    return state

- **Graph Creation**

In [ ]:
graph = StateGraph(Blog)
graph.add_node('create_outline', create_outline)
graph.add_node('write_blog', write_blog)
graph.add_node('evaluate_blog', evaluate_blog)
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'write_blog')
graph.add_edge('write_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

workflow = graph.compile()
workflow

- **Invokation**

In [ ]:
request = {"topic": "The Future of AI in Healthcare"}
response  = workflow.invoke(request)

pprint.pprint(response['blog'])
print('\n\n ---------------------------------------------------------------------------------------------------------------------- \n\n')
pprint.pprint(response['evaluation'])